# Brain Segmentation and Volumetry of T1-Weighted MRI Using FastSurfer

Structural T1-weighted MRI shows brain anatomy in fine detail. Measures such as regional volumes and cortical thickness are widely used in research and clinical neuroscience. FreeSurfer is the standard tool for extracting them, but it takes hours per scan. FastSurfer is a deep-learning alternative that segments the brain into anatomical regions in minutes on a GPU.

### Objective
The notebook is a reproducible Colab pipeline from raw data to measurements. It downloads a T1 scan from OpenNeuro (default ds000001, sub-01) over unsigned S3, with no manual upload. It installs and runs FastSurfer in one of two modes. Segmentation-only is fast (about 1–2 minutes) and gives ROI volumes. The full pipeline adds surface reconstruction (about 45–90 minutes, free FreeSurfer license required) and gives cortical thickness, surface area and gray-matter volume per region. It then parses the stats files into pandas DataFrames and saves them as CSVs.



## 1. Setup

In [ ]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                      capture_output=True, text=True).stdout
print(gpu if gpu.strip() else "No GPU detected. Go to Runtime > Change runtime type > GPU for a big speed-up.")


name, memory.total [MiB]
Tesla T4, 15360 MiB



- `RUN_SURFACE_RECON = False` → fast (~1-2 min GPU), no FreeSurfer/license needed, gives you **ROI volumes only**.
- `RUN_SURFACE_RECON = True` → slow (~45-90 min), needs a free FreeSurfer license, adds **cortical thickness, surface area, and gray matter volume per region**, plus refined volumes.

In [ ]:
#@markdown ### Segmentation only vs. full pipeline (segmentation + surface reconstruction)
RUN_SURFACE_RECON = False  #@param {type:"boolean"}

#@markdown ### OpenNeuro dataset / subject to fetch
DATASET_ID = "ds000001"   #@param {type:"string"}
SUBJECT_ID = "01"         #@param {type:"string"}
SESSION_ID = ""           #@param {type:"string"}
T1_SUFFIX  = "T1w.nii.gz" #@param {type:"string"}

SUBJECT_LABEL = f"sub-{SUBJECT_ID}"
SESSION_LABEL = f"ses-{SESSION_ID}" if SESSION_ID else ""

print("Mode:", "FULL (segmentation + surface reconstruction)" if RUN_SURFACE_RECON else "SEGMENTATION ONLY")
print("Dataset:", DATASET_ID, "| Subject:", SUBJECT_LABEL, "| Session:", SESSION_LABEL or "(none)")


Mode: SEGMENTATION ONLY
Dataset: ds000001 | Subject: sub-01 | Session: (none)


## 2. Import T1 MRI from OpenNeuro

OpenNeuro datasets are mirrored on a public S3 bucket, so we can pull just the file we need with an unsigned S3 request — no browser download, no manual upload.

We first **list** the subject's `anat/` folder so you can confirm the exact filename (BIDS naming varies a bit between datasets — some have `_run-1_`, `_acq-...`, etc. suffixes).


In [ ]:
!pip -q install awscli

anat_path = f"{DATASET_ID}/{SUBJECT_LABEL}/" + (f"{SESSION_LABEL}/" if SESSION_LABEL else "") + "anat/"
print(f"Listing s3://openneuro.org/{anat_path}")
!aws s3 ls --no-sign-request "s3://openneuro.org/{anat_path}"


Listing s3://openneuro.org/ds000001/sub-01/anat/
2020-08-21 20:24:14    5663237 sub-01_T1w.nii.gz
2020-08-21 20:24:15     669578 sub-01_inplaneT2.nii.gz


If the filename printed above doesn't match `T1_SUFFIX` from the Settings cell (e.g. it's `sub-01_run-1_T1w.nii.gz`), just edit `T1_SUFFIX` in the Settings cell — or edit `t1_filename` directly below — to match, then re-run.


In [ ]:
import os

DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

t1_filename = f"{SUBJECT_LABEL}_{T1_SUFFIX}"
s3_key = f"s3://openneuro.org/{anat_path}{t1_filename}"
local_t1 = f"{DATA_DIR}/{t1_filename}"

!aws s3 cp --no-sign-request "{s3_key}" "{local_t1}"
print("Saved to:", local_t1)


download: s3://openneuro.org/ds000001/sub-01/anat/sub-01_T1w.nii.gz to data/sub-01_T1w.nii.gz
Saved to: /content/data/sub-01_T1w.nii.gz


## 3. Install FastSurfer


In [ ]:
%cd /content
!git clone --depth 1 https://github.com/Deep-MI/FastSurfer.git
%cd /content/FastSurfer
!pip -q install -r requirements.txt

import os
os.environ["PYTHONPATH"] = "/content/FastSurfer:" + os.environ.get("PYTHONPATH", "")
os.environ["FASTSURFER_HOME"] = "/content/FastSurfer"
!chmod +x /content/FastSurfer/run_fastsurfer.sh
%cd /content
print("FastSurfer installed.")


/content
Cloning into 'FastSurfer'...
remote: Enumerating objects: 397, done.
remote: Counting objects: 100% (397/397), done.
remote: Compressing objects: 100% (359/359), done.
remote: Total 397 (delta 43), reused 192 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (397/397), 15.70 MiB | 21.91 MiB/s, done.
Resolving deltas: 100% (43/43), done.
/content/FastSurfer
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.5/117.5 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 1

## 4. FreeSurfer + license (if `RUN_SURFACE_RECON = True`)


In [ ]:
import os

FS_LICENSE_PATH = None

if RUN_SURFACE_RECON:
    FS_HOME = "/content/freesurfer"
    if not os.path.exists(FS_HOME):
        print("Downloading FreeSurfer (large file, this can take 10-15 minutes)...")
        !wget -q --show-progress https://surfer.nmr.mgh.harvard.edu/pub/dist/freesurfer/7.4.1/freesurfer-linux-centos7_x86_64-7.4.1.tar.gz -O /content/freesurfer.tar.gz
        print("Extracting...")
        !tar -xzf /content/freesurfer.tar.gz -C /content
    else:
        print("FreeSurfer already present at", FS_HOME)

    os.environ["FREESURFER_HOME"] = FS_HOME
    os.environ["PATH"] = f"{FS_HOME}/bin:" + os.environ["PATH"]

    print("\nUpload your FreeSurfer license.txt")
    print("(get one free, instantly, at https://surfer.nmr.mgh.harvard.edu/registration.html)")
    from google.colab import files
    uploaded = files.upload()
    license_name = list(uploaded.keys())[0]
    !cp "{license_name}" {FS_HOME}/license.txt
    FS_LICENSE_PATH = f"{FS_HOME}/license.txt"
    print("FreeSurfer ready, license installed at", FS_LICENSE_PATH)
else:
    print("RUN_SURFACE_RECON is False -> skipping FreeSurfer install.")


RUN_SURFACE_RECON is False -> skipping FreeSurfer install. Segmentation doesn't need it.


## 5. Run FastSurfer

- `RUN_SURFACE_RECON = False` → adds `--seg_only` (segmentation module only).
- `RUN_SURFACE_RECON = True` → adds `--fs_license` and `--parallel` (runs segmentation + recon-surf).


In [ ]:
import torch, os

OUTPUT_DIR = "/content/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
SID = SUBJECT_LABEL + (f"_{SESSION_LABEL}" if SESSION_LABEL else "")

device = "cuda" if torch.cuda.is_available() else "cpu"
threads = os.cpu_count() or 2

cmd = [
    "/content/FastSurfer/run_fastsurfer.sh",
    "--sd", OUTPUT_DIR,
    "--sid", SID,
    "--t1", local_t1,
    "--device", device,
    "--threads", str(threads),
]

if RUN_SURFACE_RECON:
    cmd += ["--fs_license", FS_LICENSE_PATH, "--parallel"]
else:
    cmd += ["--seg_only"]

# Colab runtimes always run as root, so we need to explicitly allow it
# (output files just end up owned by root, which is fine in a throwaway Colab VM)
cmd += ["--allow_root"]

full_cmd = " ".join(cmd)
print("Running:\n", full_cmd)
!{full_cmd}

Running:
 /content/FastSurfer/run_fastsurfer.sh --sd /content/output --sid sub-01 --t1 /content/data/sub-01_T1w.nii.gz --device cuda --threads 2 --seg_only --allow_root
INFO: Testing fs_time was not successful, not reporting per-command runtimes.
  running FastSurfer as root, because it will lead to files and folders created as root.
  If you are running FastSurfer in a docker container, you can specify the user
  with '-u $(id -u):$(id -g)' (see https://docs.docker.com/engine/reference/run/#user).
Start of the log for a new run_fastsurfer.sh invocation
Version: 2.6.0-dev0+96ed742
Sat Jul  4 05:10:04 AM UTC 2026

Log file for FastSurfer pipeline, run_fastsurfer.sh and segmentation(s)
Invocation: /content/FastSurfer/run_fastsurfer.sh --sd /content/output --sid sub-01 --t1 /content/data/sub-01_T1w.nii.gz --device cuda --threads 2 --seg_only --allow_root
python3 -s /content/FastSurfer/FastSurferCNN/run_prediction.py --t1 /content/data/sub-01_T1w.nii.gz --sid sub-01 --asegdkt_segfile /cont

## 6. Load the stats into pandas

- `stats/aseg+DKT.stats` (segmentation-only mode) or the refined `stats/aseg.stats` (full mode) — **ROI volumes** — always available.
- `stats/lh.aparc.stats` / `stats/rh.aparc.stats` — per-region **cortical thickness, surface area, gray matter volume** — only available when `RUN_SURFACE_RECON = True`.


In [ ]:
import pandas as pd
import os

def read_fs_stats(path):
    """Parse a FreeSurfer/FastSurfer .stats file into a DataFrame."""
    header, rows = None, []
    with open(path) as f:
        for line in f:
            if line.startswith("# ColHeaders"):
                header = line.replace("# ColHeaders", "").split()
            elif not line.startswith("#") and line.strip():
                rows.append(line.split())
    return pd.DataFrame(rows, columns=header)

stats_dir = f"{OUTPUT_DIR}/{SID}/stats"
print("Available stats files:", sorted(os.listdir(stats_dir)))

# ROI volumes -- available in BOTH modes
seg_stats_path = f"{stats_dir}/aseg.stats" if RUN_SURFACE_RECON else f"{stats_dir}/aseg+DKT.stats"
roi_volumes = read_fs_stats(seg_stats_path)
print(f"\nROI volumes from {seg_stats_path}:")
display(roi_volumes[["StructName", "Volume_mm3"]])

lh_thickness = rh_thickness = None
if RUN_SURFACE_RECON:
    lh_thickness = read_fs_stats(f"{stats_dir}/lh.aparc.stats")
    rh_thickness = read_fs_stats(f"{stats_dir}/rh.aparc.stats")
    print("\nLeft hemisphere cortical thickness / area / gray matter volume per region:")
    display(lh_thickness[["StructName", "SurfArea", "GrayVol", "ThickAvg", "ThickStd"]])


Available stats files: ['aseg+DKT.VINN.stats', 'aseg+DKT.VINN.withCC.stats', 'aseg+DKT.stats', 'aseg.VINN.stats', 'aseg.auto.mgz', 'callosum.CC.all_slices.json', 'callosum.CC.midslice.json', 'cerebellum.CerebNet.stats', 'hypothalamus.HypVINN.stats']

ROI volumes from /content/output/sub-01/stats/aseg+DKT.stats:


,StructName,Volume_mm3
0,Left-Cerebral-White-Matter,236778.336
1,Left-Lateral-Ventricle,3335.776
2,Left-Inf-Lat-Vent,146.474
3,Left-Cerebellum-White-Matter,13732.957
4,Left-Cerebellum-Cortex,48245.225
...,...,...
95,ctx-rh-superiorparietal,10426.803
96,ctx-rh-superiortemporal,13906.894
97,ctx-rh-supramarginal,9554.113
98,ctx-rh-transversetemporal,832.076


In [ ]:
# Save everything as CSVs for downstream analysis
combined_path = f"{OUTPUT_DIR}/{SID}_roi_volumes.csv"
roi_volumes.to_csv(combined_path, index=False)
print("Saved:", combined_path)

if RUN_SURFACE_RECON:
    lh_path = f"{OUTPUT_DIR}/{SID}_lh_aparc.csv"
    rh_path = f"{OUTPUT_DIR}/{SID}_rh_aparc.csv"
    lh_thickness.to_csv(lh_path, index=False)
    rh_thickness.to_csv(rh_path, index=False)
    print("Saved:", lh_path)
    print("Saved:", rh_path)


Saved: /content/output/sub-01_roi_volumes.csv
